# CNN, LSTM, and GRU with tinygrad / OpenCL

Compare binary fall/non-fall classifiers on the three current `amplitude_sequence_v2` exports. Every model receives all **96 Hz** samples: 192, 384, or 576 steps for 2, 4, or 6 seconds. Receiver views are synchronized and fused as feature channels. Inputs preserve full time resolution; recurrent training propagates gradients over the entire sequence.

The split holds out attempts, stratified by dataset and activity, with approximately 60% training, 20% validation, and 20% test groups. All overlapping windows and durations from an attempt stay together. The reported results do not establish participant- or session-independent performance. Fall targets follow action cues, not impact annotations; centered Hampel filtering uses future samples.

Install Python dependencies with `uv sync --inexact --extra analysis --extra tinygrad-legacy --extra notebook`. Select the project `.venv` kernel. On this Fedora/AMD machine, install `mesa-libOpenCL.x86_64` and enable `RUSTICL_ENABLE=radeonsi`. The notebook requires OpenCL and reports runtime failures without switching backends. Each fresh run creates a separate timestamped results directory.

GPU jobs run in separate workers capped at 4 GB RAM, without swap, and four CPU cores. Training uses 32-window effective batches and recurrent microbatches with 32-step recomputation; state gradients cross every boundary. Evaluation replays bounded forward schedules and masks extra batch slots. This preserves full-resolution inputs and full BPTT.

A **pilot** is an optional, stratified subset run used to check the training/evaluation path and estimate runtime. It does not gate or change full-data training, and its metrics do not predict full-data accuracy. `RUN_PILOT` and `RUN_FULL_TRAINING` are independent controls; both default to `False`. The pilot selects four windows per split, duration, dataset, activity, and target stratum, then profiles three epochs and a separate full-data loading pass. Full-data training uses the settings in each model section below. Existing cell outputs may be from previous runs and remain historical until their cells are executed again.

## Setup and experiment settings

Run top to bottom from a fresh kernel. Set RUN_PILOT and RUN_FULL_TRAINING independently in the setup cell. Each model has its own training settings. A seed fixes initialization and batch order; identical results across driver versions are not guaranteed.

In [4]:
import hashlib
import os
import sys
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path
from time import perf_counter

os.environ["DEV"] = "CL"
os.environ.setdefault("RUSTICL_ENABLE", "radeonsi")

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/fall_detection").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import polars as pl
from IPython.display import display

from fall_detection.analysis.sequence_training_data import (
    DURATIONS,
    LABELS,
    SegmentStore,
    fit_normalizer,
    load_sequence_index,
    split_sequence_attempts,
)
from fall_detection.analysis.sequence_training_profile import (
    prepare_profile,
    profile_table,
    refresh_projections,
)
from fall_detection.analysis.tinygrad_training import (
    TrainingConfig,
    comparison_table,
    plot_results,
    write_json,
)

RUN_PILOT = False
RUN_FULL_TRAINING = False
PILOT_WINDOWS_PER_STRATUM = 4
SPLIT_SEED = 42
DATA_ROOT = PROJECT_ROOT / "data/processed/amplitude_sequence_v2"
RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
RESULT_DIR = PROJECT_ROOT / "data/processed/tinygrad_binary_v1" / RUN_ID
RESULT_DIR.mkdir(parents=True, exist_ok=False)
print("Results:", RESULT_DIR)
print("Python:", sys.executable)
from fall_detection.analysis.sequence_training_worker import run_worker

## Index, fixed attempt split, and coverage

Read existing export indexes and manifests. Raw recordings remain unchanged; each window indexes a memory-mapped segment signal. Window counts differ from independent attempt counts. No coverage threshold or class resampling is applied.

In [5]:
preparation_started = perf_counter()
index, sources = load_sequence_index(DATA_ROOT)
index = split_sequence_attempts(index, SPLIT_SEED)
index.write_parquet(RESULT_DIR / "window_split.parquet")
provenance = {
    "sources": sources,
    "split_seed": SPLIT_SEED,
    "split_unit": "session + attempt, shared across durations",
    "split_sha256": hashlib.sha256(
        (RESULT_DIR / "window_split.parquet").read_bytes()
    ).hexdigest(),
    "code_sha256": {
        name: hashlib.sha256(
            (PROJECT_ROOT / "src/fall_detection/analysis" / name).read_bytes()
        ).hexdigest()
        for name in (
            "sequence_training_data.py",
            "tinygrad_training.py",
            "sequence_training_worker.py",
            "sequence_training_profile.py",
        )
    },
}
write_json(
    RESULT_DIR / "manifest.json",
    {
        "provenance": provenance,
        "labels": LABELS,
        "limitation": "Held-out attempts; cue-centered window labels; offline centered filtering; no participant/session holdout.",
    },
)
display(
    index.group_by("dataset", "split")
    .agg(pl.col("group").n_unique().alias("attempts"))
    .sort("dataset", "split")
)
display(
    index.group_by("duration", "split", "target")
    .len()
    .sort("duration", "split", "target")
)
coverage = (
    index.group_by("dataset", "duration", "split")
    .agg(
        pl.len().alias("windows"),
        pl.col("coverage").min().alias("minimum"),
        pl.col("coverage").quantile(0.1).alias("p10"),
        pl.col("coverage").median().alias("median"),
        pl.col("coverage").mean().alias("mean"),
    )
    .sort("dataset", "duration", "split")
)
coverage.write_csv(RESULT_DIR / "coverage.csv")
display(coverage)
metadata_preparation_seconds = perf_counter() - preparation_started

dataset,split,attempts
str,str,u32
"""akmal""","""test""",46
"""akmal""","""train""",138
"""akmal""","""validation""",47
"""continuous_non_fall""","""test""",2
"""continuous_non_fall""","""train""",8
"""continuous_non_fall""","""validation""",2
"""khaelano""","""test""",47
"""khaelano""","""train""",139
"""khaelano""","""validation""",46


duration,split,target,len
i64,str,i64,u32
2,"""test""",0,1386
2,"""test""",1,139
2,"""train""",0,4751
2,"""train""",1,382
2,"""validation""",0,1397
…,…,…,…
6,"""test""",1,54
6,"""train""",0,1399
6,"""train""",1,159


dataset,duration,split,windows,minimum,p10,median,mean
str,i64,str,u32,f64,f64,f64,f64
"""akmal""",2,"""test""",450,0.175446,0.710805,0.806813,0.775836
"""akmal""",2,"""train""",1364,0.156207,0.677487,0.806803,0.765005
"""akmal""",2,"""validation""",473,0.309938,0.699766,0.808131,0.776673
"""akmal""",4,"""test""",208,0.435427,0.728757,0.802394,0.778019
"""akmal""",4,"""train""",627,0.245267,0.691061,0.799814,0.767012
…,…,…,…,…,…,…,…
"""khaelano""",4,"""train""",631,0.481346,0.790854,0.808759,0.801204
"""khaelano""",4,"""validation""",210,0.644839,0.78044,0.80879,0.800399
"""khaelano""",6,"""test""",135,0.637806,0.762461,0.808564,0.796579


## Verify OpenCL in a bounded worker

When either run mode is enabled, the OpenCL preflight and GPU work run in a separate systemd scope with a hard 4 GB memory cap, no swap, a four-core CPU quota, and a runtime limit. If containment is unavailable or fails, the notebook stops. There is no direct-kernel fallback.

In [6]:
if RUN_PILOT or RUN_FULL_TRAINING:
    runtime = run_worker(RESULT_DIR, "probe")
    write_json(RESULT_DIR / "runtime.json", runtime)
    display(runtime)
else:
    print(
        "OpenCL preflight skipped; enable RUN_PILOT or RUN_FULL_TRAINING to run tinygrad."
    )

## Stratified subset preparation

Select four windows per existing split, duration, dataset, activity, and target stratum with seed 42. Attempts retain their original split. Fit pilot normalizers exclusively on the selected training windows; held-out signals never enter these statistics. Inputs retain all 96 Hz steps and all receiver/subcarrier features. The bounded cache and one ready batch avoid materializing a full dataset.

In [7]:
if RUN_PILOT:
    pilot_selection = prepare_profile(RESULT_DIR, PILOT_WINDOWS_PER_STRATUM, SPLIT_SEED)
    display(pl.DataFrame(pilot_selection["subset_counts"]))
else:
    print("Pilot disabled. Set RUN_PILOT=True to prepare the stratified subset.")

preparation_seconds = {}


def run_duration(kind, duration, config):
    if duration not in preparation_seconds:
        started = perf_counter()
        store = SegmentStore()
        normalizer = fit_normalizer(index.filter(pl.col("duration") == duration), store)
        normalizer.save(RESULT_DIR / f"normalizer-{duration}s.npz")
        store.clear()
        preparation_seconds[duration] = (
            metadata_preparation_seconds + perf_counter() - started
        )
        write_json(RESULT_DIR / "preparation_seconds.json", preparation_seconds)
    return run_worker(
        RESULT_DIR, "train", kind=kind, duration=duration, config=asdict(config)
    )

## Optional pilot: subset training and runtime projection

When RUN_PILOT=True, run the real training, validation, checkpoint restore, test evaluation, and synchronized inference path for all nine model/duration combinations. Three epochs separate schedule construction from repeat execution; pilot batch-one inference uses eight test windows. Measure a full-data CPU loader pass separately, without model training, because scattered subsets can have different page-cache behavior from complete passes. Combine repeat-batch compute with the measured loading term; report ideal overlap and serial execution bounds. These are sensitivity bounds, not confidence intervals. Dataset size cannot establish a stopping epoch or predict accuracy/F1. Disk-cache behavior, thermal state, and final-batch shapes can change full-run rates.

In [8]:
profile_results = []
if RUN_PILOT:
    PROFILE_CONFIG = TrainingConfig(
        batch_size=32, micro_batch_size=32, recurrent_chunk_steps=32
    )
    for duration in DURATIONS:
        for kind in ("cnn", "lstm", "gru"):
            profile_results.append(
                run_worker(
                    RESULT_DIR,
                    "profile",
                    kind=kind,
                    duration=duration,
                    config=asdict(PROFILE_CONFIG),
                    timeout_seconds=900,
                )
            )
        # One read-only CPU pass measures full-data loading; no full-data training.
        run_worker(RESULT_DIR, "loading", duration=duration, timeout_seconds=600)
        profile_results = refresh_projections(RESULT_DIR)
        projection = profile_table(profile_results)
        display(projection)
    print(
        "Projected 30-epoch training total (excluding preparation and final evaluation):",
        f"{projection['full_30_epochs_minutes'].sum() / 60:.2f} hours",
    )
else:
    print("Pilot disabled; full-data training is controlled separately.")

## CNN

Two same-padding temporal convolutions (64 channels; kernels 5 and 3), ReLU, temporal mean, and a binary linear head.

Adam trains with training-only positive-class weighting and global gradient clipping. The earliest highest validation fall F1 selects the checkpoint; patience is five epochs. Test windows do not select checkpoints or thresholds.

In [9]:
CNN_CONFIG = TrainingConfig(
    seed=42,
    epochs=30,
    batch_size=32,
    learning_rate=0.001,
    patience=5,
    hidden_size=64,
    clip_norm=1.0,
)
display(asdict(CNN_CONFIG))

In [10]:
cnn_results = []
if RUN_FULL_TRAINING:
    for duration in DURATIONS:
        cnn_results.append(run_duration("cnn", duration, CNN_CONFIG))
        display(comparison_table(cnn_results))
else:
    print(
        "Full-data training is disabled. Set RUN_FULL_TRAINING=True to enable it; "
        "pilot execution is controlled independently by RUN_PILOT."
    )

## LSTM

One 64-unit LSTM layer over every time step; the final hidden state feeds a binary linear head.

Adam trains with training-only positive-class weighting and global gradient clipping. The earliest highest validation fall F1 selects the checkpoint; patience is five epochs. Test windows do not select checkpoints or thresholds.

For 2-second full runs, use 64-step chunks; set `LSTM_2S_CHUNK_STEPS = 32` for the original shorter-setup configuration. Other durations and GRU retain 32-step chunks. This setting is independent of whether the pilot is run.

In [11]:
from dataclasses import replace

LSTM_2S_CHUNK_STEPS = 64

LSTM_CONFIG = TrainingConfig(
    seed=42,
    epochs=30,
    batch_size=32,
    learning_rate=0.001,
    patience=5,
    hidden_size=64,
    clip_norm=1.0,
)
display(asdict(LSTM_CONFIG))

In [12]:
lstm_results = []
if RUN_FULL_TRAINING:
    for duration in DURATIONS:
        config = (
            replace(LSTM_CONFIG, recurrent_chunk_steps=LSTM_2S_CHUNK_STEPS)
            if duration == 2
            else LSTM_CONFIG
        )
        lstm_results.append(run_duration("lstm", duration, config))
        display(comparison_table(lstm_results))
else:
    print(
        "Full-data training is disabled. Set RUN_FULL_TRAINING=True to enable it; "
        "pilot execution is controlled independently by RUN_PILOT."
    )

## GRU

One 64-unit reset-after GRU layer over every time step; the final hidden state feeds a binary linear head.

Adam trains with training-only positive-class weighting and global gradient clipping. The earliest highest validation fall F1 selects the checkpoint; patience is five epochs. Test windows do not select checkpoints or thresholds.

In [ ]:
GRU_CONFIG = TrainingConfig(
    seed=42,
    epochs=30,
    batch_size=32,
    learning_rate=0.001,
    patience=5,
    hidden_size=64,
    clip_norm=1.0,
)
display(asdict(GRU_CONFIG))

In [14]:
gru_results = []
if RUN_FULL_TRAINING:
    for duration in DURATIONS:
        gru_results.append(run_duration("gru", duration, GRU_CONFIG))
        display(comparison_table(gru_results))
else:
    print(
        "Full-data training is disabled. Set RUN_FULL_TRAINING=True to enable it; "
        "pilot execution is controlled independently by RUN_PILOT."
    )

## Compare accuracy, precision, recall, F1, and time

Precision, recall, and F1 refer to the positive fall class at probability ≥ 0.5. Undefined metrics are zero; per-class reports and confusion matrices include supports. Continuous non-fall false-positive proportions are reported separately.

Phase timings include per-step durations, loader wait, validation, checkpoint writes, final evaluation, and first-call inference. Training seconds include batch loading, transfers, compilation, epoch validation, and best-checkpoint writes through early stopping; initial data preparation and final evaluation are separate. Inference uses batch size one, five warm-ups, and up to 100 fixed test windows (eight in the subset pilot). Device-resident forward latency includes host dispatch and synchronization. Transfer-inclusive latency additionally covers input transfer, sigmoid, and probability retrieval; neither includes disk reads or normalization. First batch-one call latency is saved separately. These timings do not measure a live preprocessing pipeline.

In [15]:
if RUN_FULL_TRAINING:
    results = [
        result
        for name in ("cnn_results", "lstm_results", "gru_results")
        for result in globals().get(name, [])
    ]
    results_path = RESULT_DIR / "comparison.csv"
    output_dir = RESULT_DIR
elif RUN_PILOT:
    results = profile_results
    results_path = RESULT_DIR / "profile/model_metrics.csv"
    output_dir = RESULT_DIR / "profile"
    print("SUBSET PILOT METRICS: these are not full-dataset model-quality results.")
else:
    results = []
    print("No training results selected. Enable RUN_PILOT or RUN_FULL_TRAINING.")

if results:
    comparison = comparison_table(results)
    comparison.write_csv(results_path)
    display(comparison)
    for result in results:
        print(
            result["model"].upper(),
            f"{result['duration_seconds']}s",
            "test per-class metrics:",
            result["test"]["per_class"],
        )
    plot_results(results, output_dir)
    print(f"Completed {len(results)}/9 configurations. Saved to {RESULT_DIR}")

## Reload a saved model

Reconstruct the checkpoint and saved normalization in a memory-limited worker. The worker checks the first test window against its saved prediction.

In [16]:
LOAD_MODEL = "gru"
LOAD_WINDOW_SECONDS = 4
if RUN_FULL_TRAINING or RUN_PILOT:
    model_dir = RESULT_DIR if RUN_FULL_TRAINING else RESULT_DIR / "profile"
    loaded_prediction = run_worker(
        model_dir,
        "predict",
        kind=LOAD_MODEL,
        duration=LOAD_WINDOW_SECONDS,
    )
    display(loaded_prediction)
else:
    print("Run the pilot or full-data training before reloading a model.")